# 03 — Compute coverage

Intersects every branch isochrone with every census tract and records what
fraction of the tract falls inside.

| | |
|---|---|
| **Reads** | `clean/population_demographics.csv`, `features/isochrones.parquet` |
| **Writes** | `results/coverage_details.parquet` |
| **Needs** | `cpl-s3` |

Areas are measured on the WGS84 ellipsoid with `pyproj.Geod` rather than by
reprojecting, so "40% of this tract" means 40% of its real surface area.

### Running this as an Elyra node

Drag this notebook onto a pipeline canvas, then open its **node properties**:

| Property | Value |
|---|---|
| Runtime image | Datascience with Python 3.12 (or any image with pip access) |
| Kubernetes secret | `cpl-s3` → `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET` |

The notebook installs its own dependencies in the first cell, so any runtime
image works. Everything it reads and writes lives in S3 — nothing is passed
between nodes as a file, so node ordering is the only dependency.

In [ ]:
%pip install -q --index-url https://pypi.org/simple shapely pyproj geopandas

In [ ]:
import io
import os
import logging

import boto3
import pandas as pd

logging.basicConfig(level=logging.INFO, force=True)
log = logging.getLogger("compute_coverage")

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
BUCKET = os.environ["AWS_S3_BUCKET"]
log.info("Bucket: %s", BUCKET)

## Parameters

The isochrone cache is append-only and never forgets, so a mode/time pair you
have stopped fetching keeps winning the "newest row" lookup forever. This set
is what decides which combinations are live.

In [ ]:
import geopandas as gpd
from pyproj import Geod
from shapely import wkt

geod = Geod(ellps="WGS84")

ACTIVE_COMBOS = {("driving", 5), ("cycling", 10), ("walking", 10)}

## Load census tracts and measure them

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="clean/population_demographics.csv")
tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))
tracts["geometry"] = tracts["geometry"].apply(wkt.loads)
tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4326")
log.info("Loaded %d census tracts", len(tracts))

tracts["area_sqm"] = [abs(geod.geometry_area_perimeter(g)[0]) for g in tracts["geometry"]]
tracts[["geoid", "area_sqm"]].head()

## Load the newest isochrone per branch/mode/time

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="features/isochrones.parquet")
iso_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
iso_df["event_timestamp"] = pd.to_datetime(iso_df["event_timestamp"], utc=True)

idx = iso_df.groupby(["branch", "mode", "travel_time"])["event_timestamp"].idxmax()
iso_df = iso_df.loc[idx].reset_index(drop=True)

before = len(iso_df)
iso_df = iso_df[
    [(m, int(t)) in ACTIVE_COMBOS for m, t in zip(iso_df["mode"], iso_df["travel_time"])]
].reset_index(drop=True)
log.info("Active-combo filter: %d -> %d isochrones", before, len(iso_df))
log.info("Using %d isochrones across %d modes and %d travel times",
         len(iso_df), iso_df["mode"].nunique(), iso_df["travel_time"].nunique())

iso_df.groupby(["mode", "travel_time"]).size()

## Intersect

Roughly 80 branches × 870 tracts per combination. Expect a few minutes.

In [ ]:
rows = []
for i, iso in iso_df.iterrows():
    iso_geom = wkt.loads(iso["geometry_wkt"])

    for _, tract in tracts.iterrows():
        if not iso_geom.intersects(tract["geometry"]):
            continue
        intersection = iso_geom.intersection(tract["geometry"])
        inter_area = abs(geod.geometry_area_perimeter(intersection)[0])
        overlap = inter_area / tract["area_sqm"] if tract["area_sqm"] > 0 else 0

        rows.append({
            "geoid": tract["geoid"],
            "branch": iso["branch"],
            "mode": iso["mode"],
            "travel_time": iso["travel_time"],
            "overlap_proportion": overlap,
            "intersection_wkt": intersection.wkt,
        })

    if (i + 1) % 25 == 0:
        log.info("Processed %d/%d isochrones", i + 1, len(iso_df))

coverage = pd.DataFrame(rows)
log.info("Computed %d tract-branch intersections", len(coverage))
coverage.head()

## Write

In [ ]:
buf = io.BytesIO()
coverage.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="results/coverage_details.parquet", Body=buf.getvalue())
log.info("Wrote s3://%s/results/coverage_details.parquet", BUCKET)